# Evaluation: Evaluation Nodes
This notebook evaluates all `evaluation_*` node outputs against benchmark ground truth and includes a rerun utility with optional dependency execution (e.g., `modelling_node`).

In [11]:
# Setup, benchmark loading, normalization, and baseline comparison
import os
import sys
import json
import re
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from utils.dsrp_state import DSRPState

# Resolve project root robustly from notebook location
ROOT = Path.cwd().resolve()
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PILOT = ROOT / "Pilot_Evaluation"
BENCHMARK_FILE = PILOT / "DATA_sample_10" / "Data Science Research Process (DSRP) Framework.csv"
RESULTS_FOLDER = PILOT / "pilot_study_results"
VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

print(f"Project root: {ROOT}")
print(f"Benchmark file exists: {BENCHMARK_FILE.exists()}")
print(f"Results folder exists: {RESULTS_FOLDER.exists()}")

GROUPS = {
    "evaluation_metrics": [
        "evaluation_strategy",
        "learning_type",
        "problem_type",
        "evaluation_metrics_present",
        "validation_procedure",
        "effect_size_reported",
        "assumption_checks_reported",
    ],
    "evaluation_theoretical_orientation": [
        "explicit_theory",
        "implicit_theory_detected",
        "epistemological_orientation",
        "primary_research_orientation",
    ],
    "evaluation_interpretability": [
        "interpretability_discussed",
        "interpretability_approach",
        "model_transparency_level",
    ],
    "evaluation_ethical_social": [
        "privacy_protection_reported",
        "bias_fairness_considered",
        "societal_impact_discussed",
        "ethical_reflection_level",
    ],
}

ALL_EVAL_DIMENSIONS = [d for dims in GROUPS.values() for d in dims]

LIST_DIMENSIONS = {
    "learning_type",
    "problem_type",
    "validation_procedure",
}

DIMENSION_CONFIG = {
    "evaluation_strategy": {
        "group": "evaluation_metrics",
        "gt_col": "evaluation_strategy",
        "pred_parent": "evaluation_metrics",
        "pred_keys": ["evaluation_strategy"],
    },
    "learning_type": {
        "group": "evaluation_metrics",
        "gt_col": "learning_type",
        "pred_parent": "evaluation_metrics",
        "pred_keys": ["learning_type"],
    },
    "problem_type": {
        "group": "evaluation_metrics",
        "gt_col": "problem_type",
        "pred_parent": "evaluation_metrics",
        "pred_keys": ["problem_type"],
    },
    "evaluation_metrics_present": {
        "group": "evaluation_metrics",
        "gt_col": "evaluation_metrics_present",
        "pred_parent": "evaluation_metrics",
        "pred_keys": ["evaluation_metrics_present"],
    },
    "validation_procedure": {
        "group": "evaluation_metrics",
        "gt_col": "validation_procedure",
        "pred_parent": "evaluation_metrics",
        "pred_keys": ["validation_procedure"],
    },
    "effect_size_reported": {
        "group": "evaluation_metrics",
        "gt_col": "effect_size_reported",
        "pred_parent": "evaluation_metrics",
        "pred_keys": ["effect_size_reported"],
    },
    "assumption_checks_reported": {
        "group": "evaluation_metrics",
        "gt_col": "assumption_checks_reported",
        "pred_parent": "evaluation_metrics",
        "pred_keys": ["assumption_checks_reported"],
    },
    "explicit_theory": {
        "group": "evaluation_theoretical_orientation",
        "gt_col": "explicit_theory",
        "pred_parent": "evaluation_theoretical_orientation",
        "pred_keys": ["explicit_theory"],
    },
    "implicit_theory_detected": {
        "group": "evaluation_theoretical_orientation",
        "gt_col": "implicit_theory_detected",
        "pred_parent": "evaluation_theoretical_orientation",
        "pred_keys": ["implicit_theory_detected"],
    },
    "epistemological_orientation": {
        "group": "evaluation_theoretical_orientation",
        "gt_col": "epistemological_orientation",
        "pred_parent": "evaluation_theoretical_orientation",
        "pred_keys": ["epistemological_orientation"],
    },
    "primary_research_orientation": {
        "group": "evaluation_theoretical_orientation",
        "gt_col": "primary_research_orientation",
        "pred_parent": "evaluation_theoretical_orientation",
        "pred_keys": ["primary_research_orientation"],
    },
    "interpretability_discussed": {
        "group": "evaluation_interpretability",
        "gt_col": "interpretability_discussed",
        "pred_parent": "evaluation_interpretability",
        "pred_keys": ["interpretability_discussed"],
    },
    "interpretability_approach": {
        "group": "evaluation_interpretability",
        "gt_col": "interpretability_approach",
        "pred_parent": "evaluation_interpretability",
        "pred_keys": ["interpretability_approach", "interpretability_method"],
    },
    "model_transparency_level": {
        "group": "evaluation_interpretability",
        "gt_col": "model_transparency_level",
        "pred_parent": "evaluation_interpretability",
        "pred_keys": ["model_transparency_level"],
    },
    "privacy_protection_reported": {
        "group": "evaluation_ethical_social",
        "gt_col": "Privacy_protection_reported",
        "pred_parent": "evaluation_ethical_social",
        "pred_keys": ["privacy_protection_reported"],
    },
    "bias_fairness_considered": {
        "group": "evaluation_ethical_social",
        "gt_col": "bias_fairness_considered",
        "pred_parent": "evaluation_ethical_social",
        "pred_keys": ["bias_fairness_considered"],
    },
    "societal_impact_discussed": {
        "group": "evaluation_ethical_social",
        "gt_col": "societal_impact_discussed",
        "pred_parent": "evaluation_ethical_social",
        "pred_keys": ["societal_impact_discussed"],
    },
    "ethical_reflection_level": {
        "group": "evaluation_ethical_social",
        "gt_col": "Ethical Reflection Level",
        "pred_parent": "evaluation_ethical_social",
        "pred_keys": ["ethical_reflection_level"],
    },
}


def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None


def _split_multi(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if isinstance(value, list):
        items = value
    else:
        text = str(value).strip()
        if not text:
            return []

        if text.startswith("[") and text.endswith("]"):
            text = text[1:-1].strip()

        items = re.split(r"[;,]", text)

    out = []
    for item in items:
        t = normalize_text(item)
        if t:
            out.append(t)
    return out


def _canonical_yes_no(value):
    text = normalize_text(value)
    if not text:
        return "No"
    return "Yes" if text.lower() in {"yes", "true", "1", "y"} else "No"


def _canonical_effect_assumption(value):
    text = normalize_text(value)
    if not text:
        return "Not reported"
    mapping = {
        "reported": "Reported",
        "not reported": "Not reported",
        "not applicable": "Not applicable",
        "not_applicable": "Not applicable",
    }
    return mapping.get(text.lower(), text)


def _canonical_eval_strategy(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        "statistical model diagnostics": "Statistical Model Diagnostics",
        "machine learning evaluation": "Machine Learning Evaluation",
        "mixed evaluation": "Mixed Evaluation",
    }
    return mapping.get(text.lower(), text)


def _canonical_learning_type(value):
    mapping = {
        "supervised": "Supervised",
        "supervised learning": "Supervised",
        "unsupervised": "Unsupervised",
        "unsupervised learning": "Unsupervised",
        "reinforcement": "Reinforcement",
        "reinforcement learning": "Reinforcement",
        "optimization-based learning": "Optimization-Based Learning",
        "optimisation-based learning": "Optimization-Based Learning",
        "not applicable": "Not applicable",
        "not_applicable": "Not applicable",
    }
    values = _split_multi(value)
    out = []
    for item in values:
        canonical = mapping.get(item.lower(), item)
        if canonical not in out:
            out.append(canonical)
    return sorted(out)


def _canonical_problem_type(value):
    mapping = {
        "classification": "Classification",
        "regression": "Regression",
        "clustering": "Clustering",
        "dimensionality reduction": "Dimensionality Reduction",
        "dimensionality_reduction": "Dimensionality Reduction",
        "reward optimisation": "Reward Optimisation",
        "reward optimization": "Reward Optimisation",
        "optimisation": "Optimisation",
        "optimization": "Optimisation",
        "not applicable": "Not applicable",
        "not_applicable": "Not applicable",
    }
    values = _split_multi(value)
    out = []
    for item in values:
        canonical = mapping.get(item.lower(), item)
        if canonical not in out:
            out.append(canonical)
    return sorted(out)


def _canonical_validation_procedure(value):
    mapping = {
        "not reported": "Not reported",
        "train/test split": "train/test split",
        "cross-validation": "cross-validation",
        "hold-out dataset": "hold-out dataset",
        "temporal split": "temporal split",
    }
    values = _split_multi(value)
    out = []
    for item in values:
        canonical = mapping.get(item.lower(), item)
        if canonical not in out:
            out.append(canonical)

    if not out:
        out = ["Not reported"]

    return sorted(out)


def _canonical_epistemic(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        "theory-driven research": "Theory-driven research",
        "hybrid (data + theory)": "Hybrid (data + theory)",
        "data-driven discovery": "Data-driven discovery",
    }
    return mapping.get(text.lower(), text)


def _canonical_primary_orientation(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        "method-oriented research": "Method-oriented research",
        "knowledge-oriented research": "Knowledge-oriented research",
        "hybrid (method + knowledge)": "Hybrid (method + knowledge)",
    }
    return mapping.get(text.lower(), text)


def _canonical_interpretability_approach(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        "inherently interpretable model": "Inherently interpretable model",
        "post-hoc explanation": "Post-hoc explanation",
        "none reported": "None reported",
    }
    return mapping.get(text.lower(), text)


def _canonical_transparency(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        "high transparency": "High transparency",
        "moderate transparency": "Moderate transparency",
        "low transparency": "Low transparency",
    }
    return mapping.get(text.lower(), text)


def _canonical_ethical_level(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        "high": "High",
        "moderate": "Moderate",
        "low": "Low",
    }
    return mapping.get(text.lower(), text)


def normalize_dimension(dim, value):
    if dim == "evaluation_strategy":
        return _canonical_eval_strategy(value)
    if dim == "learning_type":
        return _canonical_learning_type(value)
    if dim == "problem_type":
        return _canonical_problem_type(value)
    if dim == "evaluation_metrics_present":
        return _canonical_yes_no(value)
    if dim == "validation_procedure":
        return _canonical_validation_procedure(value)
    if dim in {"effect_size_reported", "assumption_checks_reported"}:
        return _canonical_effect_assumption(value)
    if dim in {"explicit_theory", "implicit_theory_detected", "interpretability_discussed", "privacy_protection_reported", "bias_fairness_considered", "societal_impact_discussed"}:
        return _canonical_yes_no(value)
    if dim == "epistemological_orientation":
        return _canonical_epistemic(value)
    if dim == "primary_research_orientation":
        return _canonical_primary_orientation(value)
    if dim == "interpretability_approach":
        return _canonical_interpretability_approach(value)
    if dim == "model_transparency_level":
        return _canonical_transparency(value)
    if dim == "ethical_reflection_level":
        return _canonical_ethical_level(value)
    return value


def format_dimension_value(dim, value):
    if dim in LIST_DIMENSIONS:
        return "; ".join(value) if value else "[]"
    return str(value) if value is not None else "(missing)"


def list_jaccard(gt_list, pred_list):
    gt_set = set(gt_list or [])
    pred_set = set(pred_list or [])
    if not gt_set and not pred_set:
        return 1.0
    return len(gt_set & pred_set) / len(gt_set | pred_set)


def load_agent_results(results_folder: Path):
    agent_data = {}
    for paper_dir in sorted(results_folder.glob("*/")):
        if not paper_dir.is_dir():
            continue
        results_file = paper_dir / "aggregated" / "results.json"
        if not results_file.exists():
            continue
        with open(results_file, "r", encoding="utf-8") as f:
            parsed = json.load(f)
            paper_id = normalize_paper_id(parsed.get("paper_id", paper_dir.name))
            dsrp_outputs = parsed.get("dsrp_outputs", {})
            if paper_id:
                agent_data[paper_id] = dsrp_outputs
    return agent_data


def extract_prediction(dsrp_outputs, dim):
    cfg = DIMENSION_CONFIG[dim]
    parent_obj = dsrp_outputs.get(cfg["pred_parent"], {})
    if not isinstance(parent_obj, dict):
        parent_obj = {}

    raw_value = None
    for key in cfg["pred_keys"]:
        if key in parent_obj:
            raw_value = parent_obj.get(key)
            break

    return normalize_dimension(dim, raw_value)


benchmark_df = pd.read_csv(BENCHMARK_FILE)
benchmark_df.columns = benchmark_df.columns.str.strip()

required_cols = ["Paper ID", "Gatekeeper"] + [cfg["gt_col"] for cfg in DIMENSION_CONFIG.values()]
missing_cols = [c for c in required_cols if c not in benchmark_df.columns]
if missing_cols:
    raise ValueError(f"Missing required benchmark columns: {missing_cols}")

benchmark_eval = benchmark_df[required_cols].copy()
benchmark_eval["Paper ID"] = benchmark_eval["Paper ID"].apply(normalize_paper_id)
benchmark_eval["Gatekeeper"] = benchmark_eval["Gatekeeper"].astype(str).str.strip().str.lower()

for dim, cfg in DIMENSION_CONFIG.items():
    benchmark_eval[cfg["gt_col"]] = benchmark_eval[cfg["gt_col"]].apply(lambda v: normalize_dimension(dim, v))

benchmark_eval = benchmark_eval[benchmark_eval["Gatekeeper"] == "include"].copy()
benchmark_eval = benchmark_eval.dropna(subset=["Paper ID"]).reset_index(drop=True)

print(f"Included benchmark rows: {len(benchmark_eval)}")

agent_data = load_agent_results(RESULTS_FOLDER)
print(f"Loaded aggregated outputs for {len(agent_data)} papers")

comparison_rows = []
missing_agent_results = []

for _, row in benchmark_eval.iterrows():
    paper_id = row["Paper ID"]

    if paper_id not in agent_data:
        missing_agent_results.append(paper_id)
        continue

    outputs = agent_data[paper_id]
    record = {"Paper ID": paper_id}

    dim_match_flags = []

    for dim, cfg in DIMENSION_CONFIG.items():
        gt_value = normalize_dimension(dim, row[cfg["gt_col"]])
        pred_value = extract_prediction(outputs, dim)
        is_match = gt_value == pred_value
        dim_match_flags.append(is_match)

        record[f"GT_{dim}"] = format_dimension_value(dim, gt_value)
        record[f"Agent_{dim}"] = format_dimension_value(dim, pred_value)
        record[f"Match_{dim}"] = "MATCH" if is_match else "MISMATCH"

        if dim in LIST_DIMENSIONS:
            record[f"Jaccard_{dim}"] = list_jaccard(gt_value, pred_value)

    for group_name, group_dims in GROUPS.items():
        group_match = all(record[f"Match_{d}"] == "MATCH" for d in group_dims)
        record[f"Match_{group_name}"] = "MATCH" if group_match else "MISMATCH"

    record["Match_All_Eval_Dimensions"] = "MATCH" if all(dim_match_flags) else "MISMATCH"
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)
print(f"Comparable rows: {len(comparison_df)}")

if missing_agent_results:
    print(f"Missing agent outputs for {len(missing_agent_results)} papers: {', '.join(missing_agent_results)}")

if len(comparison_df):
    show_cols = ["Paper ID"] + [f"Match_{g}" for g in GROUPS] + ["Match_All_Eval_Dimensions"]
    print(comparison_df[show_cols].to_string(index=False))
else:
    print("No comparable rows found.")

Project root: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow
Benchmark file exists: True
Results folder exists: True
Included benchmark rows: 7
Loaded aggregated outputs for 10 papers
Comparable rows: 7
 Paper ID Match_evaluation_metrics Match_evaluation_theoretical_orientation Match_evaluation_interpretability Match_evaluation_ethical_social Match_All_Eval_Dimensions
 2018 - 8                 MISMATCH                                 MISMATCH                          MISMATCH                           MATCH                  MISMATCH
2019 - 33                 MISMATCH                                    MATCH                          MISMATCH                           MATCH                  MISMATCH
 2020 - 8                 MISMATCH                                 MISMATCH                          MISMATCH                           MATCH                  MISMATCH
2021 - 28                 MISMATCH                                 MISMATCH                          

In [12]:
# Dimension-wise metrics and group-level agreement
if len(comparison_df) == 0:
    raise ValueError("No rows available for evaluation.")

metrics_rows = []

for dim in ALL_EVAL_DIMENSIONS:
    y_true = comparison_df[f"GT_{dim}"].tolist()
    y_pred = comparison_df[f"Agent_{dim}"].tolist()

    acc = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    row = {
        "group": DIMENSION_CONFIG[dim]["group"],
        "dimension": dim,
        "samples": len(y_true),
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    }

    if dim in LIST_DIMENSIONS:
        row["avg_jaccard"] = comparison_df[f"Jaccard_{dim}"].mean()

    metrics_rows.append(row)

    print("=" * 96)
    print(f"DIMENSION: {dim} ({DIMENSION_CONFIG[dim]['group']})")
    print("=" * 96)
    print(f"accuracy............ {acc:.4f}")
    print(f"precision_macro..... {precision_macro:.4f}")
    print(f"recall_macro........ {recall_macro:.4f}")
    print(f"f1_macro............ {f1_macro:.4f}")
    if dim in LIST_DIMENSIONS:
        print(f"avg_jaccard......... {comparison_df[f'Jaccard_{dim}'].mean():.4f}")

    report = classification_report(
        y_true,
        y_pred,
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).T
    print("\nPer-label report:")
    print(report_df.to_string())

metrics_df = pd.DataFrame(metrics_rows)

print("\n" + "=" * 96)
print("SUMMARY TABLE")
print("=" * 96)
print(metrics_df.to_string(index=False))

print("\nGROUP-LEVEL EXACT AGREEMENT")
for group_name in GROUPS:
    col = f"Match_{group_name}"
    group_match_count = int((comparison_df[col] == "MATCH").sum())
    overall_total = len(comparison_df)
    print(f"{group_name}: {group_match_count}/{overall_total} ({100 * group_match_count / overall_total:.1f}%)")

overall_match_count = int((comparison_df["Match_All_Eval_Dimensions"] == "MATCH").sum())
overall_total = len(comparison_df)
print(f"\nOverall exact agreement across all evaluation dimensions: {overall_match_count}/{overall_total} ({100 * overall_match_count / overall_total:.1f}%)")

DIMENSION: evaluation_strategy (evaluation_metrics)
accuracy............ 0.7143
precision_macro..... 0.8333
recall_macro........ 0.8333
f1_macro............ 0.7778

Per-label report:
                               precision    recall  f1-score   support
Machine Learning Evaluation     1.000000  0.500000  0.666667  4.000000
Mixed Evaluation                0.500000  1.000000  0.666667  2.000000
Statistical Model Diagnostics   1.000000  1.000000  1.000000  1.000000
accuracy                        0.714286  0.714286  0.714286  0.714286
macro avg                       0.833333  0.833333  0.777778  7.000000
weighted avg                    0.857143  0.714286  0.714286  7.000000
DIMENSION: learning_type (evaluation_metrics)
accuracy............ 0.5714
precision_macro..... 0.4444
recall_macro........ 0.4167
f1_macro............ 0.4286
avg_jaccard......... 0.6429

Per-label report:
                          precision    recall  f1-score   support
Not applicable             1.000000  1.000000  1.

In [13]:
# Detailed mismatch diagnostics
mismatch_df = comparison_df[comparison_df["Match_All_Eval_Dimensions"] == "MISMATCH"].copy()
print(f"Total mismatched papers: {len(mismatch_df)}")

if len(mismatch_df):
    detail_cols = [
        "Paper ID",
        *[f"Match_{g}" for g in GROUPS],
        "Match_All_Eval_Dimensions",
    ]
    print(mismatch_df[detail_cols].to_string(index=False))

    # Show full side-by-side labels for faster debugging
    long_cols = ["Paper ID"]
    for dim in ALL_EVAL_DIMENSIONS:
        long_cols.extend([f"GT_{dim}", f"Agent_{dim}", f"Match_{dim}"])

    print("\nFull mismatch detail:")
    print(mismatch_df[long_cols].to_string(index=False))
else:
    print("No mismatches found.")

Total mismatched papers: 7
 Paper ID Match_evaluation_metrics Match_evaluation_theoretical_orientation Match_evaluation_interpretability Match_evaluation_ethical_social Match_All_Eval_Dimensions
 2018 - 8                 MISMATCH                                 MISMATCH                          MISMATCH                           MATCH                  MISMATCH
2019 - 33                 MISMATCH                                    MATCH                          MISMATCH                           MATCH                  MISMATCH
 2020 - 8                 MISMATCH                                 MISMATCH                          MISMATCH                           MATCH                  MISMATCH
2021 - 28                 MISMATCH                                 MISMATCH                          MISMATCH                           MATCH                  MISMATCH
2021 - 56                 MISMATCH                                 MISMATCH                          MISMATCH                        

## Iteration Utility
Use the helper below to rerun evaluation nodes on selected papers. It can run `modelling_node` first so `evaluation_metrics` receives the modelling context it depends on.

In [14]:
import importlib

import nodes.modelling_node as modelling_module
import nodes.evaluation_nodes.evaluation_metrics_foundational as eval_metrics_module
import nodes.evaluation_nodes.evaluation_theoretical_orientation as eval_theory_module
import nodes.evaluation_nodes.evaluation_interpretability as eval_interp_module
import nodes.evaluation_nodes.evaluation_ethical_social as eval_ethical_module


def run_evaluation_iteration(
    paper_ids,
    llm_model="gpt-4o-mini",
    run_modelling_dependency=True,
    verbose=True,
):
    """Rerun evaluation nodes on selected papers and return compact comparison results."""

    importlib.reload(modelling_module)
    importlib.reload(eval_metrics_module)
    importlib.reload(eval_theory_module)
    importlib.reload(eval_interp_module)
    importlib.reload(eval_ethical_module)

    modelling_node = modelling_module.modelling_node
    evaluation_metrics_foundational_node = eval_metrics_module.evaluation_metrics_foundational_node
    evaluation_theoretical_orientation_node = eval_theory_module.evaluation_theoretical_orientation_node
    evaluation_interpretability_node = eval_interp_module.evaluation_interpretability_node
    evaluation_ethical_social_node = eval_ethical_module.evaluation_ethical_social_node

    benchmark_lookup = benchmark_eval.set_index("Paper ID")
    rows = []
    raw_outputs = {}

    original_dir = Path.cwd().resolve()
    prev_env_model = os.getenv("DSRP_LLM_MODEL")

    os.chdir(ROOT)
    os.environ["DSRP_LLM_MODEL"] = llm_model

    try:
        for paper_id in paper_ids:
            pid = normalize_paper_id(paper_id)
            if not pid:
                continue

            if pid not in benchmark_lookup.index:
                rows.append({"Paper ID": pid, "Error": "Paper ID not found in benchmark include set"})
                continue

            if verbose:
                print(f"Running evaluation pipeline for {pid} with model={llm_model}...")

            state = DSRPState(
                paper_id=pid,
                dsrp_outputs={},
                collection_name=COLLECTION_NAME,
                persist_directory=str(VECTOR_DB_PATH),
                embedding_model=EMBEDDING_MODEL,
                gatekeeper={},
            )

            try:
                if run_modelling_dependency:
                    out = modelling_node(state)
                    state["dsrp_outputs"] = out.get("dsrp_outputs", state["dsrp_outputs"])

                out = evaluation_metrics_foundational_node(state)
                state["dsrp_outputs"] = out.get("dsrp_outputs", state["dsrp_outputs"])

                out = evaluation_theoretical_orientation_node(state)
                state["dsrp_outputs"] = out.get("dsrp_outputs", state["dsrp_outputs"])

                out = evaluation_interpretability_node(state)
                state["dsrp_outputs"] = out.get("dsrp_outputs", state["dsrp_outputs"])

                out = evaluation_ethical_social_node(state)
                state["dsrp_outputs"] = out.get("dsrp_outputs", state["dsrp_outputs"])

                raw_outputs[pid] = state["dsrp_outputs"]

                record = {"Paper ID": pid, "Model": llm_model}
                dim_match_flags = []

                for dim, cfg in DIMENSION_CONFIG.items():
                    gt_value = normalize_dimension(dim, benchmark_lookup.loc[pid, cfg["gt_col"]])
                    pred_value = extract_prediction(state["dsrp_outputs"], dim)
                    is_match = gt_value == pred_value
                    dim_match_flags.append(is_match)

                    record[f"GT_{dim}"] = format_dimension_value(dim, gt_value)
                    record[f"New_Agent_{dim}"] = format_dimension_value(dim, pred_value)
                    record[f"Match_{dim}"] = "MATCH" if is_match else "MISMATCH"

                for group_name, group_dims in GROUPS.items():
                    group_match = all(record[f"Match_{d}"] == "MATCH" for d in group_dims)
                    record[f"Match_{group_name}"] = "MATCH" if group_match else "MISMATCH"

                record["Match_All_Eval_Dimensions"] = "MATCH" if all(dim_match_flags) else "MISMATCH"
                rows.append(record)

                if verbose:
                    print("  " + " | ".join(f"{g}: {record[f'Match_{g}']}" for g in GROUPS))
                    print(f"  Match_All_Eval_Dimensions: {record['Match_All_Eval_Dimensions']}")

            except Exception as e:
                err_row = {"Paper ID": pid, "Model": llm_model, "Error": str(e)}
                rows.append(err_row)
                if verbose:
                    print(f"  Error: {e}")

    finally:
        if prev_env_model is None:
            os.environ.pop("DSRP_LLM_MODEL", None)
        else:
            os.environ["DSRP_LLM_MODEL"] = prev_env_model
        os.chdir(original_dir)

    retest_df = pd.DataFrame(rows)

    if len(retest_df):
        print("\n" + "-" * 120)
        print("RE-TEST RESULTS (compact summary)")
        print("-" * 120)

        show_cols = [
            "Paper ID",
            "Model",
            *[f"Match_{g}" for g in GROUPS],
            "Match_All_Eval_Dimensions",
        ]
        cols = [c for c in show_cols if c in retest_df.columns]
        if cols:
            print(retest_df[cols].to_string(index=False))
        else:
            print(retest_df.to_string(index=False))

    return retest_df, raw_outputs

## Iteration 1
Rerun evaluation nodes on baseline-mismatched papers.

In [15]:
papers_to_rerun = comparison_df[comparison_df["Match_All_Eval_Dimensions"] == "MISMATCH"]["Paper ID"].tolist()
print(f"Papers selected for rerun: {len(papers_to_rerun)}")

retest_df, retest_outputs = run_evaluation_iteration(
    paper_ids=papers_to_rerun,
    llm_model="gpt-4o-mini",
    run_modelling_dependency=True,
    verbose=True,
)
retest_df

Papers selected for rerun: 7
Running evaluation pipeline for 2018 - 8 with model=gpt-4o-mini...
  evaluation_metrics: MISMATCH | evaluation_theoretical_orientation: MISMATCH | evaluation_interpretability: MISMATCH | evaluation_ethical_social: MATCH
  Match_All_Eval_Dimensions: MISMATCH
Running evaluation pipeline for 2019 - 33 with model=gpt-4o-mini...
  evaluation_metrics: MISMATCH | evaluation_theoretical_orientation: MATCH | evaluation_interpretability: MISMATCH | evaluation_ethical_social: MATCH
  Match_All_Eval_Dimensions: MISMATCH
Running evaluation pipeline for 2020 - 8 with model=gpt-4o-mini...
  evaluation_metrics: MISMATCH | evaluation_theoretical_orientation: MISMATCH | evaluation_interpretability: MISMATCH | evaluation_ethical_social: MATCH
  Match_All_Eval_Dimensions: MISMATCH
Running evaluation pipeline for 2021 - 28 with model=gpt-4o-mini...
  evaluation_metrics: MISMATCH | evaluation_theoretical_orientation: MISMATCH | evaluation_interpretability: MISMATCH | evaluation_

,Paper ID,Model,GT_evaluation_strategy,New_Agent_evaluation_strategy,Match_evaluation_strategy,GT_learning_type,New_Agent_learning_type,Match_learning_type,GT_problem_type,New_Agent_problem_type,...,New_Agent_societal_impact_discussed,Match_societal_impact_discussed,GT_ethical_reflection_level,New_Agent_ethical_reflection_level,Match_ethical_reflection_level,Match_evaluation_metrics,Match_evaluation_theoretical_orientation,Match_evaluation_interpretability,Match_evaluation_ethical_social,Match_All_Eval_Dimensions
0,2018 - 8,gpt-4o-mini,Machine Learning Evaluation,Mixed Evaluation,MISMATCH,Supervised,Supervised,MATCH,Classification; Regression,Classification; Regression,...,No,MATCH,Low,Low,MATCH,MISMATCH,MISMATCH,MISMATCH,MATCH,MISMATCH
1,2019 - 33,gpt-4o-mini,Statistical Model Diagnostics,Statistical Model Diagnostics,MATCH,Not applicable,Not applicable,MATCH,Not applicable,Not applicable,...,No,MATCH,Low,Low,MATCH,MISMATCH,MATCH,MISMATCH,MATCH,MISMATCH
2,2020 - 8,gpt-4o-mini,Machine Learning Evaluation,Machine Learning Evaluation,MATCH,Unsupervised,Unsupervised,MATCH,Clustering; Dimensionality Reduction,Clustering,...,No,MATCH,Low,Low,MATCH,MISMATCH,MISMATCH,MISMATCH,MATCH,MISMATCH
3,2021 - 28,gpt-4o-mini,Mixed Evaluation,Mixed Evaluation,MATCH,Supervised,[],MISMATCH,Regression,Regression,...,No,MATCH,Low,Low,MATCH,MISMATCH,MISMATCH,MISMATCH,MATCH,MISMATCH
4,2021 - 56,gpt-4o-mini,Mixed Evaluation,Mixed Evaluation,MATCH,Supervised; Unsupervised,Supervised; Unsupervised,MATCH,Clustering; Regression,Clustering; Regression,...,No,MATCH,Low,Low,MATCH,MISMATCH,MATCH,MISMATCH,MATCH,MISMATCH
5,2023 - 58,gpt-4o-mini,Machine Learning Evaluation,Mixed Evaluation,MISMATCH,Supervised,Unsupervised,MISMATCH,Clustering,Clustering,...,No,MATCH,Low,Low,MATCH,MISMATCH,MISMATCH,MISMATCH,MATCH,MISMATCH
6,2024 - 99,gpt-4o-mini,Machine Learning Evaluation,Machine Learning Evaluation,MATCH,Supervised,Optimization-Based Learning,MISMATCH,Reward Optimisation,Reward Optimisation,...,No,MISMATCH,Moderate,Low,MISMATCH,MISMATCH,MISMATCH,MISMATCH,MISMATCH,MISMATCH


## Watch Reasoning for Single Paper

In [18]:
# Inspect one paper in detail: label-by-label comparison + node reasoning

def _extract_reasoning_block(node_output):
    if not isinstance(node_output, dict):
        return "(no reasoning field found)", ""

    reasoning = (
        node_output.get("validated_reasoning")
        or node_output.get("reasoning")
        or node_output.get("reasoning_explanation")
        or "(no reasoning field found)"
    )
    audit = node_output.get("audit_commentary", "")
    return str(reasoning), str(audit or "")


def _extract_bibliography(node_output):
    if not isinstance(node_output, dict):
        return []
    bib = node_output.get("validated_bibliography")
    if isinstance(bib, list):
        return bib
    bib = node_output.get("bibliography")
    if isinstance(bib, list):
        return bib
    return []


def inspect_single_paper_reasoning(retest_df, retest_outputs, paper_id, show_bibliography=False):
    pid = normalize_paper_id(paper_id)

    if "Paper ID" not in retest_df.columns:
        raise ValueError("retest_df does not contain 'Paper ID'.")

    rows = retest_df[retest_df["Paper ID"].apply(normalize_paper_id) == pid]
    if len(rows) == 0:
        raise ValueError(f"Paper ID not found in retest_df: {paper_id}")

    row = rows.iloc[-1].to_dict()
    outputs = retest_outputs.get(pid, {})

    print("\n" + "=" * 120)
    print(f"PAPER: {row.get('Paper ID', '(missing)')} | MODEL: {row.get('Model', '(missing)')}")
    print("=" * 120)
    print(f"Overall Match (all evaluation dimensions): {row.get('Match_All_Eval_Dimensions', '(missing)')}")

    for group_name, dims in GROUPS.items():
        print("\n" + "-" * 120)
        print(f"GROUP: {group_name} | Match: {row.get(f'Match_{group_name}', '(missing)')}")
        print("-" * 120)

        for dim in dims:
            print(f"{dim}")
            print(f"  GT    : {row.get(f'GT_{dim}', '(missing)')}")
            print(f"  Agent : {row.get(f'New_Agent_{dim}', '(missing)')}")
            print(f"  Match : {row.get(f'Match_{dim}', '(missing)')}")

        node_output = outputs.get(group_name, {})
        reasoning, audit = _extract_reasoning_block(node_output)
        print("\nReasoning:")
        print(reasoning)

        if audit.strip():
            print("\nAudit Commentary:")
            print(audit)

        if show_bibliography:
            bibliography = _extract_bibliography(node_output)
            print("\nBibliography:")
            if not bibliography:
                print("  (none)")
            else:
                for i, item in enumerate(bibliography, start=1):
                    page = item.get("page", "") if isinstance(item, dict) else ""
                    section = item.get("section", "") if isinstance(item, dict) else ""
                    quote = item.get("direct_quote", "") if isinstance(item, dict) else ""
                    print(f"  [{i}] page={page} | section={section}")
                    print(f"      quote: {quote}")


# Example usage after running Iteration 1 cell:
if "retest_df" in globals() and "retest_outputs" in globals() and len(retest_df):
    example_paper_id = retest_df.iloc[0]["Paper ID"]
    inspect_single_paper_reasoning(
        retest_df=retest_df,
        retest_outputs=retest_outputs,
        paper_id=example_paper_id,
        show_bibliography=False,
    )
else:
    print("Run the Iteration 1 cell first so retest_df and retest_outputs are available.")


PAPER: 2018 - 8 | MODEL: gpt-4o-mini
Overall Match (all evaluation dimensions): MISMATCH

------------------------------------------------------------------------------------------------------------------------
GROUP: evaluation_metrics | Match: MISMATCH
------------------------------------------------------------------------------------------------------------------------
evaluation_strategy
  GT    : Machine Learning Evaluation
  Agent : Mixed Evaluation
  Match : MISMATCH
learning_type
  GT    : Supervised
  Agent : Supervised
  Match : MATCH
problem_type
  GT    : Classification; Regression
  Agent : Classification; Regression
  Match : MATCH
evaluation_metrics_present
  GT    : Yes
  Agent : Yes
  Match : MATCH
validation_procedure
  GT    : train/test split
  Agent : train/test split
  Match : MATCH
effect_size_reported
  GT    : Not applicable
  Agent : Not applicable
  Match : MATCH
assumption_checks_reported
  GT    : Not applicable
  Agent : Not applicable
  Match : MATCH

R

In [19]:
# All-paper comparison per evaluation node (group-wise)

def show_all_papers_per_node(source_df=None, include_label_details=True):
    """Display paper-by-paper comparison grouped by evaluation node."""

    if source_df is None:
        if "retest_df" in globals() and isinstance(retest_df, pd.DataFrame) and len(retest_df):
            source_df = retest_df.copy()
            value_prefix = "New_Agent_"
            print("Using rerun table: retest_df")
        elif "comparison_df" in globals() and isinstance(comparison_df, pd.DataFrame) and len(comparison_df):
            source_df = comparison_df.copy()
            value_prefix = "Agent_"
            print("Using baseline table: comparison_df")
        else:
            raise ValueError("No populated dataframe found. Run baseline or iteration cells first.")
    else:
        source_df = source_df.copy()
        value_prefix = "New_Agent_" if any(c.startswith("New_Agent_") for c in source_df.columns) else "Agent_"

    if "Paper ID" not in source_df.columns:
        raise ValueError("Expected 'Paper ID' column not found in source dataframe.")

    for group_name, dims in GROUPS.items():
        print("\n" + "=" * 120)
        print(f"NODE: {group_name}")
        print("=" * 120)

        summary_col = f"Match_{group_name}"
        if summary_col in source_df.columns:
            match_count = int((source_df[summary_col] == "MATCH").sum())
            total = len(source_df)
            print(f"Node-level exact match: {match_count}/{total} ({100 * match_count / total:.1f}%)")

        cols = ["Paper ID"]
        for dim in dims:
            gt_col = f"GT_{dim}"
            pred_col = f"{value_prefix}{dim}"
            match_col = f"Match_{dim}"

            if gt_col in source_df.columns:
                cols.append(gt_col)
            if pred_col in source_df.columns:
                cols.append(pred_col)
            if match_col in source_df.columns:
                cols.append(match_col)

        display_df = source_df[cols].copy()

        if include_label_details:
            print(display_df.to_string(index=False))
        else:
            compact_cols = ["Paper ID"] + [f"Match_{d}" for d in dims if f"Match_{d}" in display_df.columns]
            print(display_df[compact_cols].to_string(index=False))


# Run report
show_all_papers_per_node(include_label_details=True)

Using rerun table: retest_df

NODE: evaluation_metrics
Node-level exact match: 0/7 (0.0%)
 Paper ID        GT_evaluation_strategy New_Agent_evaluation_strategy Match_evaluation_strategy         GT_learning_type     New_Agent_learning_type Match_learning_type                      GT_problem_type     New_Agent_problem_type Match_problem_type GT_evaluation_metrics_present New_Agent_evaluation_metrics_present Match_evaluation_metrics_present          GT_validation_procedure                                                                                                                                   New_Agent_validation_procedure Match_validation_procedure GT_effect_size_reported New_Agent_effect_size_reported Match_effect_size_reported GT_assumption_checks_reported New_Agent_assumption_checks_reported Match_assumption_checks_reported
 2018 - 8   Machine Learning Evaluation              Mixed Evaluation                  MISMATCH               Supervised                  Supervised       

## Targeted Single-Paper Debug
Use this cell to inspect one specific paper quickly.

In [16]:
single_retest_df, single_outputs = run_evaluation_iteration(
    paper_ids=["2018 - 8"],  # Change paper ID here
    llm_model="gpt-4o-mini",
    run_modelling_dependency=True,
    verbose=True,
)
single_retest_df

Running evaluation pipeline for 2018 - 8 with model=gpt-4o-mini...


KeyboardInterrupt: 

## Full-Set Rerun (Optional)
Run on all included benchmark papers when you want a full refresh after node or prompt changes.

In [ ]:
all_papers = comparison_df["Paper ID"].tolist()
print(f"Total included papers: {len(all_papers)}")

full_retest_df, full_retest_outputs = run_evaluation_iteration(
    paper_ids=all_papers,
    llm_model="gpt-4o-mini",
    run_modelling_dependency=True,
    verbose=True,
)
full_retest_df